In [1]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




answers    0
label      0
dtype: int64
0


,answers,label
36676,"['Car sickness, also known as motion sickness,...",1
32108,"[""In the United States, the two major politica...",1
6227,['Eating sub optimally does not mean a swift c...,0
35885,['A randomize function is a function in progra...,1
16566,['The sample complexity of a machine learning ...,0
10667,['The makers of adblocking software often grac...,0
27190,"[""Osmosis is a natural process that occurs whe...",1
26554,"[""If the Earth stopped spinning, a lot of thin...",1
40184,['A software agent is a program that performs ...,1
28494,['Feeling good is a sensation that is usually ...,1


In [2]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x   # IMPORTANT: fallback safe

    return str(x)



In [3]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

,answers,label
0,"basically there are many categories of "" best ...",0
1,salt is good for not dying in car crashes and ...,0
2,the way it works is that old tv stations got a...,0
3,you ca n't just go around assassinating the le...,0
4,wanting to kill the shit out of germans drives...,0


In [4]:
df_feat = df.copy()
df_feat.sample(5)

,answers,label
7537,[ google maps calculates distance based on an ...,0
14652,a nation is a group of people with several fac...,0
25446,free web browsers like opera generate income t...,1
25116,encryption is a way of hiding information so t...,1
43965,to remove the effect of the overall stock mark...,1


In [6]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")

In [8]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [9]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [10]:
def extract_features(text):
    doc = nlp(text)

    tokens = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            "chat_word_count": 0,
            "punct_count": 0,
            "function_count": 0,
            "discourse_count": 0,
            "spelling_error_ratio": 0,
            "ttr": 0,
            "function_word_ratio": 0,
            "discourse_ratio": 0,
            "sentence_length": 0,
            "avg_word_length": 0,
        }

    chat_word_count = sum(1 for t in tokens if t in chat_words)
    punct_count = sum(1 for t in doc if t.is_punct)

    function_count = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    spelling_errors = sum(1 for t in doc if t.is_alpha and t.is_oov)

    return {
        "chat_word_count": chat_word_count,
        "punct_count": punct_count,
        "function_count": function_count,
        "discourse_count": discourse_count,
        "spelling_error_ratio": spelling_errors / total_alpha,
        "ttr": len(set(alpha_tokens)) / total_alpha,
        "function_word_ratio": function_count / total_tokens,
        "discourse_ratio": discourse_count / total_tokens,
        "sentence_length": total_tokens,
        "avg_word_length": sum(len(t) for t in alpha_tokens) / total_alpha,
    }

In [11]:
def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]

    if len(feats) == 0:
        return {}

    df = pd.DataFrame(feats)

    return {
        "chat_word_count": df["chat_word_count"].sum(),
        "punct_count": df["punct_count"].sum(),

        "function_count": df["function_count"].sum(),
        "discourse_count": df["discourse_count"].sum(),

        "spelling_error_ratio": df["spelling_error_ratio"].mean(),
        "ttr": df["ttr"].mean(),
        "function_word_ratio": df["function_word_ratio"].mean(),
        "discourse_ratio": df["discourse_ratio"].mean(),

        "sentence_length_mean": df["sentence_length"].mean(),
        "sentence_length_std": df["sentence_length"].std(),

        "avg_word_length": df["avg_word_length"].mean(),
    }

In [12]:
def build_final_features(df):
    rows = []

    for _, row in df.iterrows():
        text = row["answers"]
        label = row["label"]

        # 1. split into sentences
        sentences = split_sentences(text)

        # 2. aggregate sentence-level features → document-level features
        features = aggregate_sentence_features(sentences)

        # 3. attach label
        features["label"] = label

        rows.append(features)

    return pd.DataFrame(rows)

In [13]:
df_final = build_final_features(df_feat)

In [14]:
import joblib
joblib.dump(df_final, "final_features.pkl")

['final_features.pkl']